# Aggregating

In this notebook you will learn how to

- merge products or industries into broader groups
- sum a SUT over one or more dimensions
- compute price layer rates

## Load the data

See [01_getting_started](01_getting_started.ipynb) for an explanation of these steps.

In [ ]:
from pathlib import Path

import pandas as pd
import sutlab as sl

data_folder = Path("data")
years = [2016, 2017, 2018, 2019]

metadata = sl.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

current_paths = [data_folder / f"ta_l_{year}.parquet" for year in years]
sut = sl.load_sut_from_separated_parquet(years, current_paths, metadata, "current_year")

## Merge products

An aggregation needs a **mapping**: a table with the old code in `from` and the new code in `to`. We merge all the drinks into one product. `V22` also contains vinegar (`V220900`), which we leave out.

In [ ]:
drinks_codes = sut.get_codes_products(["V22*", "~V220900"], as_list=True)

product_mapping = pd.DataFrame({
    "from": drinks_codes,
    "to": "V22",
})
product_mapping.head()

The new code needs a label. We give it in a table with the same columns as the classification.

In [ ]:
product_labels = pd.DataFrame({
    "nrnr": ["V22"],
    "nrnr_txt": ["Drikkevarer"],
})

`full_coverage=False` means that products outside the mapping are kept as they are. By default all products must be in the mapping, so that you do not forget any by accident.

In [ ]:
sut_drinks = sut.aggregate_classification_products(
    product_mapping,
    metadata=product_labels,
    full_coverage=False,
)
sut_drinks.get_codes_products("V22*")

The drinks have become one product, and the vinegar is still there. The result is a SUT like any other, so it can be inspected.

In [ ]:
sut_drinks.inspect_products("V22").balance

## Merge industries

Industries work in the same way. Here we merge the food industries. There are similar functions for transactions and for the consumption groups.

In [ ]:
food_codes = sut.get_codes_industries("1000*", as_list=True)

industry_mapping = pd.DataFrame({
    "from": food_codes,
    "to": "100000",
})
industry_labels = pd.DataFrame({
    "brch": ["100000"],
    "brch_txt": ["Fødevareindustri"],
})

sut_food = sut.aggregate_classification_industries(
    industry_mapping,
    metadata=industry_labels,
    full_coverage=False,
)
sut_food.inspect_industries("100000").balance

**For volumes:** aggregate both the SUT in current prices and the SUT in previous year's prices first, and compute the chained volumes afterwards. Chained volumes cannot be added together.

## Totals

`compute_totals` sums over everything except the dimensions you name. The year is always kept. The result is an ordinary DataFrame.

In [ ]:
transaction_totals = sut.compute_totals("transaction")
transaction_totals.head(10)

Combine it with `filter_rows` to sum over part of the data. Here it is household consumption by consumption group.

In [ ]:
households = sut.filter_rows(transactions="3110")
household_totals = households.compute_totals("category")
household_totals.head(10)

## Price layer rates

`compute_price_layer_rates` gives the rate of each price layer. Each layer is divided by the basic price plus the layers before it. Here are the rates for household consumption of drinks in each year.

In [ ]:
drinks_households = sut_drinks.filter_rows(products="V22", transactions="3110")
drinks_households.compute_price_layer_rates(["product", "transaction"])

---
Back to the [overview](README.md)